# 00 — TransformerLens: A Library Tour

Before any interpretability concepts: what is this library, and how do you
drive it? This notebook is pure mechanics — loading a model, tokenizing,
running it, reading outputs, and the hook system's shape — with zero mech
interp theory. Notebook 01 builds the concepts on top of exactly what you
see working here.

Nothing below is guessed — every call in this notebook was run against a
live model before being written down.


In [1]:
import torch
from transformer_lens import HookedTransformer, utils

torch.set_grad_enabled(False)  # we're only doing inference in this whole track, never training
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


/home/keqingsimp/.conda/envs/mech-interp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


/tmp/ipykernel_104992/3466895314.py:2: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  from transformer_lens import HookedTransformer, utils


## Loading a model

`HookedTransformer.from_pretrained(name)` downloads (and caches locally
after the first run) a model's weights from Hugging Face and wraps them in
TransformerLens's instrumented implementation. It works for a long list of
open models (GPT-2 family, Pythia, Llama, Mistral, Qwen, ...) — you're not
limited to GPT-2, that's just the small, fast one we're using for exercises.


In [2]:
model = HookedTransformer.from_pretrained("gpt2", device=device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 18311.42it/s]

Loaded pretrained model gpt2 into HookedTransformer


## Tokenization: text <-> tokens

Four methods cover almost everything you need:

- `model.to_tokens(text)` — string -> tensor of token ids, shape `[batch, pos]`
- `model.to_str_tokens(text)` — string -> list of the individual token *strings* (useful for labeling plots)
- `model.to_string(tokens)` — tokens -> string, the inverse of `to_tokens`
- `model.to_single_token(text)` — string -> a single token id, and it'll error loudly if `text` isn't exactly one token (a good sanity check when you assume something is one token and want to be told if you're wrong)

By default `to_tokens` prepends a beginning-of-sequence token
(`<|endoftext|>` for GPT-2) — pass `prepend_bos=False` if you don't want
that.


In [3]:
tokens = model.to_tokens("hello world")
print("with BOS:   ", tokens)
print("without BOS:", model.to_tokens("hello world", prepend_bos=False))
print("as strings: ", model.to_str_tokens("hello world"))
print("round trip: ", model.to_string(tokens))


with BOS:    tensor([[50256, 31373,   995]], device='cuda:0')
without BOS: tensor([[31373,   995]], device='cuda:0')
as strings:  ['<|endoftext|>', 'hello', ' world']
round trip:  ['<|endoftext|>hello world']


## Running the model

Two ways to run a forward pass, and the difference matters:

- `model(tokens)` — a plain forward pass. Fast, returns only the final
  logits. Use this when you just want the model's output and don't care
  about internals.
- `model.run_with_cache(tokens)` — runs the same forward pass but also
  records every intermediate activation. Slower and more memory, but this is
  the one mech interp actually runs on almost every call, since the whole
  point is inspecting internals.


In [4]:
logits = model(tokens)
print("plain forward — logits shape:", logits.shape)  # [batch, pos, d_vocab]

logits, cache = model.run_with_cache(tokens)
print("run_with_cache — same logits shape:", logits.shape, "  plus a cache with", len(cache), "entries")


plain forward — logits shape: torch.Size([1, 3, 50257])
run_with_cache — same logits shape: torch.Size([1, 3, 50257])   plus a cache with 210 entries


## Reading the cache

`cache` is a dict-like `ActivationCache`. Two equivalent ways to index it:

- Full hook name: `cache["blocks.0.attn.hook_pattern"]`
- Shorthand tuple: `cache["pattern", 0]` — TransformerLens infers the
  `blocks.{layer}.attn.hook_` prefix for you. Handy, but only works for
  activation names it recognizes this way; when in doubt, use the full name.

`utils.get_act_name("pattern", 0)` builds the full name string
programmatically — prefer this over hand-typing hook name strings in real
code, since it can't typo.


In [5]:
full_name = "blocks.0.attn.hook_pattern"
shorthand = cache["pattern", 0]
via_name = cache[full_name]
via_util = cache[utils.get_act_name("pattern", 0)]

print(torch.equal(shorthand, via_name), torch.equal(via_name, via_util))
print(shorthand.shape)  # [batch, head, dest_pos, src_pos]


True True
torch.Size([1, 12, 3, 3])


## Generating text

`model.generate` does autoregressive sampling for you — pass a string or
tokens, get a string or tokens back. `do_sample=False` makes it
deterministic (greedy decoding), which is what you want for reproducible
debugging.


In [6]:
completion = model.generate("The capital of France is", max_new_tokens=5, do_sample=False, verbose=False)
print(completion)


The capital of France is now home to the world


## `utils.test_prompt` — the fastest sanity-check tool

Give it a prompt and an expected answer; it tells you the answer's rank and
probability, plus what the model actually would have said instead. This is
the tool to reach for first whenever you're about to build a whole
experiment around "the model predicts X here" — check that claim in one
line before writing ten lines of analysis on top of it.


In [7]:
utils.test_prompt(
    "The Eiffel Tower is in the city of",
    " Paris",
    model,
    prepend_space_to_answer=False,
)


Tokenized prompt: ['<|endoftext|>', 'The', ' E', 'iff', 'el', ' Tower', ' is', ' in', ' the', ' city', ' of']
Tokenized answer: [' Paris']


Performance on answer token:
Rank: 1        Logit: 15.34 Prob:  6.87% Token: | Paris|

Top 0th token. Logit: 15.50 Prob:  8.05% Token: | London|
Top 1th token. Logit: 15.34 Prob:  6.87% Token: | Paris|
Top 2th token. Logit: 14.56 Prob:  3.14% Token: | Amsterdam|
Top 3th token. Logit: 14.36 Prob:  2.58% Token: | New|
Top 4th token. Logit: 14.32 Prob:  2.48% Token: | Berlin|
Top 5th token. Logit: 13.86 Prob:  1.57% Token: | Hamburg|
Top 6th token. Logit: 13.79 Prob:  1.46% Token: | L|
Top 7th token. Logit: 13.74 Prob:  1.38% Token: | Brussels|
Top 8th token. Logit: 13.73 Prob:  1.37% Token: | Cologne|
Top 9th token. Logit: 13.65 Prob:  1.27% Token: | E|


Ranks of the answer tokens: [(' Paris', 1)]

## Hooks, at the mechanism level

A **hook** is a named tap point inserted into the forward pass. You've only
*read* from them so far (via `run_with_cache`); you can also intercept and
*edit* the value flowing through one, which is how causal experiments like
activation patching work later on.

`model.run_with_hooks(tokens, fwd_hooks=[(name, fn)])` runs one forward pass
with `fn` inserted at the hook called `name`. `fn` receives the activation
tensor and a `hook` object (which has `.name`, so one function can be reused
across many hook points), and must return the (possibly modified) tensor.


In [8]:
def zero_out(activation, hook):
    print(f"  intercepted {hook.name}, shape {tuple(activation.shape)}")
    return torch.zeros_like(activation)

# ablate layer 0's attention output entirely and see the prediction degrade
clean_logits = model(tokens)
ablated_logits = model.run_with_hooks(tokens, fwd_hooks=[("blocks.0.attn.hook_z", zero_out)])

print("same logits?", torch.allclose(clean_logits, ablated_logits))


  intercepted blocks.0.attn.hook_z, shape (1, 3, 12, 64)
same logits? False


## Cheat sheet

| Want to... | Call |
|---|---|
| Load a model | `HookedTransformer.from_pretrained(name, device=device)` |
| Text -> tokens | `model.to_tokens(text)` |
| Tokens -> readable strings | `model.to_str_tokens(text)` |
| Tokens -> text | `model.to_string(tokens)` |
| Text -> exactly one token id | `model.to_single_token(text)` |
| Fast forward pass, logits only | `model(tokens)` |
| Forward pass + every activation | `model.run_with_cache(tokens)` |
| Read one activation | `cache[utils.get_act_name(name, layer)]` or `cache[name, layer]` |
| Generate text | `model.generate(text, max_new_tokens=n, do_sample=False)` |
| Quick "does the model predict X" check | `utils.test_prompt(prompt, answer, model)` |
| Run with an activation intercepted/edited | `model.run_with_hooks(tokens, fwd_hooks=[(name, fn)])` |
| Model architecture constants | `model.cfg.d_model`, `.n_layers`, `.n_heads`, etc. |

## Next

`01_transformer_primer.ipynb` — same tools, now used to actually explain the
residual stream, attention, and MLP conceptually.
